# Data Preparation for analysis

In [42]:
import pandas as pd
import re
import chardet
from matplotlib import pyplot as plt
import seaborn as sns
import warnings
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from datetime import date
from scipy import stats
import ipywidgets as widgets
from IPython.display import display
warnings.filterwarnings("ignore")
pd.options.display.float_format = '{:.2f}'.format

In [43]:
converted = pd.read_csv('/content/drive/MyDrive/Finn/Dataset - Data Analyst Product - Task 1 - Session data  - Channel data from converted users.csv')

In [44]:
converted.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157247 entries, 0 to 157246
Data columns (total 12 columns):
 #   Column                      Non-Null Count   Dtype 
---  ------                      --------------   ----- 
 0   final_id                    157247 non-null  int64 
 1   context_locale              157244 non-null  object
 2   event_text                  157247 non-null  object
 3   context_page_path           157244 non-null  object
 4   context_page_title          155465 non-null  object
 5   context_user_agent          157244 non-null  object
 6   timestamp                   157247 non-null  object
 7   first_campaign              157087 non-null  object
 8   first_content               57104 non-null   object
 9   first_medium                157095 non-null  object
 10  first_source                157247 non-null  object
 11  first_time_order_completed  89553 non-null   object
dtypes: int64(1), object(11)
memory usage: 14.4+ MB


I see a few columns that are in object format, i need to convert them

In [45]:
for col in ['timestamp', 'first_time_order_completed']:
    converted[col] = pd.to_datetime(converted[col], errors='coerce')

Also there are some missing values, i need to deal with them.

In [46]:
def clean_dataframe(df):

    # Remove rows with NaN in critical columns
    df = df.dropna(subset=['context_locale', 'context_page_path', 'context_user_agent', 'first_campaign', 'first_medium'])

    # Handle context_page_title
    df['context_page_title'] = df['context_page_title'].fillna('Unknown Page Title')

    #Handle first_content
    df['first_content'] = df['first_content'].fillna('Unknown Content')

    return df

converted = clean_dataframe(converted)



Another thing worth transforming is `context_user_agent`, where i can extract a lot of valueable information about session

In [47]:
def classify_page_path(path):
    if not isinstance(path, str):
        return "Other"

    path = path.lower()

    if "/subscribe" in path:
        if any(brand in path for brand in ["bmw", "tesla", "jeep", "opel", "fiat", "mini", "polestar", "skoda", "mitsubishi", "renault", "chevrolet", "dacia"]):
            return "Subscribe - Car Brand"
        elif "elektro" in path or "electric" in path or "plug-in-hybrid" in path or "benzin" in path or "gas" in path or "diesel" in path:
            return "Subscribe - Fuel Type"
        elif "automatik" in path or "automatic" in path or "manuell" in path:
            return "Subscribe - Transmission"
        elif any(duration in path for duration in ["1-monat", "6-monate", "9-monate", "12-monate", "18-monate", "24-monate"]):
            return "Subscribe - Duration"
        elif any(body in path for body in ["coupe", "limousine", "suv", "van", "kombi"]):
            return "Subscribe - Car Body"
        else:
            return "Subscribe - General"
    elif "/pdp" in path:
        return "Product Detail Page"
    elif "/checkout" in path:
        return "Checkout"
    elif "/mycars" in path or "/myaccount" in path or "/self-service" in path:
        return "User Account"
    elif any(brand in path for brand in ["bmw", "tesla", "jeep", "opel", "fiat", "mini", "polestar", "skoda", "mitsubishi", "renault", "chevrolet", "dacia"]):
        return "Brand Specific Page"
    elif "/en-us" in path or "/de-de" in path:
        return "Localized Page"
    elif "/" == path:
        return "Homepage"
    elif "/b2b" in path:
        return "B2B"
    elif any(rental in path for rental in ["longtermrental", "monthlyrent-car", "car-flatrate", "car-subscription", "flexible-leasing", "rental-subscription"]):
        return "Rental"
    elif "/contact" in path:
        return "Contact"
    else:
        return "Other"

def add_page_path_classification_column(df, input_column='context_page_path', output_column='page_path_category'):
    df[output_column] = df[input_column].apply(classify_page_path)
    return df


converted = add_page_path_classification_column(converted)


In [48]:

import pandas as pd

def extract_user_agent_info(user_agent):
    platform = None
    browser = None

    if not isinstance(user_agent, str):
        return None, None

    # Platform Extraction
    if "iPhone" in user_agent or "iPad" in user_agent:
        platform = "iOS"
    elif "Android" in user_agent:
        platform = "Android"
    elif "Macintosh" in user_agent or "Windows" in user_agent or "Linux" in user_agent:
        platform = "PC"
    else:
        platform = None

    # Browser Extraction
    if "Chrome" in user_agent:
        browser = "Chrome"
    elif "Safari" in user_agent and "Chrome" not in user_agent:
        browser = "Safari"
    elif "Firefox" in user_agent:
        browser = "Firefox"
    elif "SamsungBrowser" in user_agent:
        browser = "SamsungBrowser"
    elif "Edge" in user_agent:
        browser = "Edge"
    elif "Opera" in user_agent or "OPR" in user_agent:
        browser = "Opera"
    else:
        browser = None

    return platform, browser

def update_user_agent_columns(df, input_column='context_user_agent', output_platform='platform', output_browser='browser'):

    platform_browser_data = df[input_column].apply(extract_user_agent_info).tolist()
    platforms, browsers = zip(*platform_browser_data)

    df[output_platform] = list(platforms)
    df[output_browser] = list(browsers)

update_user_agent_columns(converted)




In [49]:
def classify_page_title(title):
    if not isinstance(title, str):
        return "Unbekannt"

    title = title.lower()

    if "bestellung" in title or "order" in title:
        return "Order/Payment"
    elif "fahrzeuge im überblick" in title or "bir bakışta araçlar" in title:
        return "Vehicle Overview"
    elif "auto abo" in title:
        return "Car Subscription"
    elif "dacia duster" in title:
        return "Dacia Duster Offer"
    elif "kombi im auto abo" in title:
        return "Kombi Car Subscription Offer"
    else:
        return "Other"

def add_page_title_classification_column(df, input_column='context_page_title', output_column='page_title_category'):
    df[output_column] = df[input_column].apply(classify_page_title)
    return df

# Füge die Klassifikationsspalte hinzu
converted = add_page_title_classification_column(converted)



In [50]:
def optimize_locale(locale):

    if not isinstance(locale, str):
        return "unknown"  # Handle non-string values

    locale = locale.lower()

    if "-" in locale:
        return locale.split("-")[0]  # Extract language code
    elif "_" in locale:
        return locale.split("_")[0] #extract language code
    else:
        return locale  # Return as is if no region code

def add_optimized_locale_column(df, input_column='context_locale', output_column='optimized_locale'):

    df[output_column] = df[input_column].apply(optimize_locale)
    return df

converted = add_optimized_locale_column(converted)


In [51]:
def classify_campaign(campaign):
    if not isinstance(campaign, str):
        return "Unknown"

    campaign = campaign.lower()

    if re.search(r's\|(?:de|us)\|p\|subs\|intent\|(finn|miete|subscription|leasing|rental|abo|finanzieren/kauf|competitor)', campaign):
        return "Intent-Based Search"
    elif re.search(r's\|(?:de|us)\|p\|subs\|carbrand(?:only|&model)?\|', campaign):
        return "Car Brand/Model Specific Search"
    elif re.search(r'ps\|(?:de|us)\|(?:b|p)\|subs\|(tofu|mofu|bofu)', campaign):
        return "Funnel Stage (ToFu/MoFu/BoFu)"
    elif re.search(r'd\|(?:de|us)\|(?:b|p)\|subs\|display', campaign):
        return "Display Remarketing/Broad"
    elif re.search(r'd\|(?:de|us)\|(?:b|p)\|subs\|discovery', campaign):
        return "Discovery Ads"
    elif re.search(r'perfmax', campaign):
        return "Performance Max Campaigns"
    elif re.search(r'tofu|lowerfunnel|bofu|mofu', campaign):
        return "Funnel Stage (Text)"
    elif re.search(r'review|comparison|productreview', campaign):
        return "Review/Comparison"
    elif re.search(r'retention|return_coordination|handover_protocol', campaign):
        return "Operational/Retention"
    elif re.search(r'finn.auto|domain_click', campaign):
        return 'Finn Specific'
    elif re.search(r'sparneuwagen|mivodo|cocos_wonderland', campaign):
        return 'Misc/Branded'
    elif re.match(r'^\d+$', campaign):
        return "Numerical ID"
    else:
        return "Other/Unclassified"

def add_campaign_classification_column(df, input_column='first_campaign', output_column='campaign_category'):
    df[output_column] = df[input_column].apply(classify_campaign)
    return df #return the modified dataframe.

# Add the classification column
converted = add_campaign_classification_column(converted)

converted.head()

,final_id,context_locale,event_text,context_page_path,context_page_title,context_user_agent,timestamp,first_campaign,first_content,first_medium,first_source,first_time_order_completed,page_path_category,platform,browser,page_title_category,optimized_locale,campaign_category
0,202315551,de-DE,Filter,/subscribe,Fahrzeuge im Überblick | FINN,Mozilla/5.0 (iPhone; CPU iPhone OS 15_3_1 like...,2022-04-02 23:43:53.195000+00:00,S|DE|P|Subs|Intent|FINN,79597880,cpc,google,2022-03-24 19:31:18.353000+00:00,Subscribe - General,iOS,Safari,Vehicle Overview,de,Intent-Based Search
1,202315551,de-DE,Filter,/subscribe,Fahrzeuge im Überblick | FINN,Mozilla/5.0 (iPhone; CPU iPhone OS 15_3_1 like...,2022-04-02 23:43:53.195000+00:00,S|DE|P|Subs|Intent|FINN,79597880,cpc,google,2022-03-24 19:31:18.353000+00:00,Subscribe - General,iOS,Safari,Vehicle Overview,de,Intent-Based Search
2,202315551,de-DE,Product List Viewed,/subscribe,Fahrzeuge im Überblick | FINN,Mozilla/5.0 (iPhone; CPU iPhone OS 15_3_1 like...,2022-04-02 23:44:20.715000+00:00,S|DE|P|Subs|Intent|FINN,79597880,cpc,google,2022-03-24 19:31:18.353000+00:00,Subscribe - General,iOS,Safari,Vehicle Overview,de,Intent-Based Search
3,202315551,de-DE,Product List Viewed,/subscribe,Fahrzeuge im Überblick | FINN,Mozilla/5.0 (iPhone; CPU iPhone OS 15_3_1 like...,2022-04-02 23:44:20.715000+00:00,S|DE|P|Subs|Intent|FINN,79597880,cpc,google,2022-03-24 19:31:18.353000+00:00,Subscribe - General,iOS,Safari,Vehicle Overview,de,Intent-Based Search
4,209571451,ru,Checkout Step Viewed,/de-DE/checkout/thank_you/209571451/8552084246...,Bestellung | Vielen Dank für Ihre Bestellung,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,2022-04-07 23:44:13.235000+00:00,S|DE|P|Subs|DSA,Unknown Content,cpc,google,2022-04-07 23:28:18.708000+00:00,Checkout,PC,Safari,Order/Payment,ru,Other/Unclassified


In [52]:
all_users = pd.read_csv('/content/drive/MyDrive/Finn/Dataset - Data Analyst Product - Task 1 - Session data  - Funnel data for all users.csv')

In [53]:
for col in ['timestamp']:
    all_users[col] = pd.to_datetime(all_users[col], errors='coerce')

In [54]:
all_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19690 entries, 0 to 19689
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   anonymous_id        19690 non-null  object             
 1   context_locale      19690 non-null  object             
 2   event_text          19690 non-null  object             
 3   context_page_path   19690 non-null  object             
 4   context_page_title  19610 non-null  object             
 5   context_user_agent  19690 non-null  object             
 6   timestamp           19674 non-null  datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), object(6)
memory usage: 1.1+ MB


In [55]:
def clean_all_users_table_revised(df):


    # Handle timestamp (remove rows with NaN)
    df = df.dropna(subset=['timestamp'])

    return df


all_users = clean_all_users_table_revised(all_users)
print(all_users.info())

<class 'pandas.core.frame.DataFrame'>
Index: 19674 entries, 0 to 19689
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   anonymous_id        19674 non-null  object             
 1   context_locale      19674 non-null  object             
 2   event_text          19674 non-null  object             
 3   context_page_path   19674 non-null  object             
 4   context_page_title  19594 non-null  object             
 5   context_user_agent  19674 non-null  object             
 6   timestamp           19674 non-null  datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), object(6)
memory usage: 1.2+ MB
None


In [56]:
all_users = add_page_title_classification_column(all_users)

In [57]:
update_user_agent_columns(all_users)

In [58]:
all_users = add_optimized_locale_column(all_users)

In [59]:
all_users = add_page_path_classification_column(all_users)

In [60]:
all_users.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19674 entries, 0 to 19689
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   anonymous_id         19674 non-null  object             
 1   context_locale       19674 non-null  object             
 2   event_text           19674 non-null  object             
 3   context_page_path    19674 non-null  object             
 4   context_page_title   19594 non-null  object             
 5   context_user_agent   19674 non-null  object             
 6   timestamp            19674 non-null  datetime64[ns, UTC]
 7   page_title_category  19674 non-null  object             
 8   platform             19640 non-null  object             
 9   browser              16999 non-null  object             
 10  optimized_locale     19674 non-null  object             
 11  page_path_category   19674 non-null  object             
dtypes: datetime64[ns, UTC](

In [61]:
converted.final_id.count()

np.int64(157084)

# Analysis

Now i need to explain my logic of further analysis:

- we have two keys in tables, but they are not matching. So, if i want to compare numbers from converted sample to all users, i need to aggregate data and then merge it on multiple categorries, that i created during this EDA stage.

In [62]:
def aggregate_session_data_optimized(df):

    if df.empty:
        return pd.DataFrame()

    # Sort the DataFrame by timestamp
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values(by='timestamp')

    # Calculate session duration
    session_duration = (df['timestamp'].max() - df['timestamp'].min()).total_seconds() / 3600

    # Aggregate session data
    aggregated_data = {
        'final_id': df['final_id'].iloc[0],
        'platform': df['platform'].iloc[0],
        'browser': df['browser'].iloc[0],
        'page_title_category': df['page_title_category'].iloc[0],
        'optimized_locale': df['optimized_locale'].iloc[0],
        'campaign_category': df['campaign_category'].iloc[0],
        'first_event_text': df['event_text'].iloc[0],
        'last_event_text': df['event_text'].iloc[-1],
        'event_count': len(df),
        'first_date': df['timestamp'].iloc[0].date(),
        'last_date': df['timestamp'].iloc[-1].date(),
        'session_duration_hours': session_duration,
        'page_path_category': df['page_path_category'].iloc[0],
        'first_time_order_completed': df['first_time_order_completed'].iloc[0].date()
    }

    return pd.DataFrame([aggregated_data])

def aggregate_all_sessions_optimized(df):


    aggregated_sessions = []
    for final_id, session_df in df.groupby('final_id'):
        aggregated_sessions.append(aggregate_session_data_optimized(session_df))

    result = pd.concat(aggregated_sessions, ignore_index=True)

    def classify_user(order_time):
        if pd.notna(order_time):
            return 'returning'
        else:
            return 'new'

    result['user_type'] = result['first_time_order_completed'].apply(classify_user)

    return result





aggregated_df = aggregate_all_sessions_optimized(converted)
aggregated_df.head()

,final_id,platform,browser,page_title_category,optimized_locale,campaign_category,first_event_text,last_event_text,event_count,first_date,last_date,session_duration_hours,page_path_category,first_time_order_completed,user_type
0,66501,PC,Chrome,Other,de,Funnel Stage (Text),Filter,Product Viewed,798,2022-04-01,2022-04-19,427.98,Subscribe - General,2022-03-21,returning
1,177451,Android,Chrome,Vehicle Overview,de,Numerical ID,Product List Viewed,Product List Viewed,28,2022-04-03,2022-04-05,49.87,Subscribe - General,2022-03-07,returning
2,318551,Android,Chrome,Car Subscription,de,Intent-Based Search,Product Viewed,Product Viewed,1,2022-04-17,2022-04-17,0.00,Product Detail Page,NaN,new
3,985701,Android,Chrome,Car Subscription,de,Review/Comparison,UserAccount,UserAccount,1,2022-04-07,2022-04-07,0.00,Homepage,NaN,new
4,1970451,PC,Chrome,Vehicle Overview,de,Numerical ID,Product List Viewed,Filter,34,2022-04-08,2022-04-08,0.04,Subscribe - General,NaN,new


In [63]:
grouped_data_converted = aggregated_df.groupby(
    ['platform', 'browser', 'optimized_locale', 'first_date']
).agg({
    'final_id': 'count',
    'session_duration_hours': 'median',
    'event_count': 'median'
}).reset_index()

grouped_data_converted = grouped_data_converted.rename(columns={
    'final_id': 'count_ids',
    'session_duration_hours': 'median_duration_hours',
    'event_count': 'median_events_per_session'
})


grouped_data_converted.sort_values(by='count_ids', ascending=False).tail()


,platform,browser,optimized_locale,first_date,count_ids,median_duration_hours,median_events_per_session
221,iOS,Safari,pt,2022-04-12,1,52.42,114.00
222,iOS,Safari,ru,2022-04-03,1,0.03,22.00
223,iOS,Safari,ru,2022-04-08,1,65.52,17.00
224,iOS,Safari,tr,2022-04-01,1,342.68,1548.00
225,iOS,Safari,zh,2022-04-04,1,0.49,38.00


In [64]:
grouped_data_converted.count_ids.sum()

np.int64(1379)

In [65]:
def aggregate_all_users_session_data(df):
    if df.empty:
        return pd.DataFrame()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values(by='timestamp')
    session_duration = (df['timestamp'].max() - df['timestamp'].min()).total_seconds() / 3600
    aggregated_data = {
        'anonymous_id': df['anonymous_id'].iloc[0],
        'platform': df['platform'].iloc[0],
        'browser': df['browser'].iloc[0],
        'page_title_category': df['page_title_category'].iloc[0],
        'optimized_locale': df['optimized_locale'].iloc[0],
        'first_event_text': df['event_text'].iloc[0],
        'last_event_text': df['event_text'].iloc[-1],
        'event_count': len(df),
        'first_date': df['timestamp'].iloc[0].date(),
        'last_date': df['timestamp'].iloc[-1].date(),
        'session_duration_hours': session_duration,
        'page_path_category': df['page_path_category'].iloc[0],
    }
    return pd.DataFrame([aggregated_data])

def aggregate_all_users_sessions(df):
    aggregated_sessions = []
    for anonymous_id, session_df in df.groupby('anonymous_id'):
        aggregated_sessions.append(aggregate_all_users_session_data(session_df))
    return pd.concat(aggregated_sessions, ignore_index=True)

aggregated_all_df = aggregate_all_users_sessions(all_users)

In [66]:
grouped_data_all_users = aggregated_all_df.groupby(['platform', 'browser', 'optimized_locale', 'first_date']).agg({'anonymous_id': 'count', 'session_duration_hours': 'median', 'event_count': 'median'}).reset_index()
grouped_data_all_users = grouped_data_all_users.rename(columns={'anonymous_id': 'count_ids', 'session_duration_hours': 'median_duration_hours', 'event_count': 'median_events_per_session'})
grouped_data_all_users.sort_values(by='count_ids', ascending=False).tail()

,platform,browser,optimized_locale,first_date,count_ids,median_duration_hours,median_events_per_session
41,iOS,Safari,fr,2022-04-12,1,0.00,5.00
44,iOS,Safari,nl,2022-04-12,1,0.03,4.00
43,iOS,Safari,ko,2022-04-12,1,0.70,22.00
42,iOS,Safari,it,2022-04-12,1,0.03,46.00
46,iOS,Safari,ru,2022-04-12,1,0.00,3.00


In [67]:
merged_df = pd.merge(grouped_data_converted, grouped_data_all_users, on=['platform', 'browser', 'optimized_locale', 'first_date'], suffixes=('_converted', '_all_users'))

In [68]:
merged_df['CR'] = merged_df['count_ids_converted'] / merged_df['count_ids_all_users']

In [69]:
merged_df

,platform,browser,optimized_locale,first_date,count_ids_converted,median_duration_hours_converted,median_events_per_session_converted,count_ids_all_users,median_duration_hours_all_users,median_events_per_session_all_users,CR
0,Android,Chrome,de,2022-04-12,14,23.19,23.50,271,0.01,9.00,0.05
1,Android,Chrome,en,2022-04-12,3,0.00,1.00,116,0.00,1.50,0.03
2,Android,Chrome,es,2022-04-12,1,0.02,9.00,4,0.00,1.50,0.25
3,Android,Chrome,ru,2022-04-12,1,0.00,1.00,5,0.04,15.00,0.20
4,PC,Chrome,de,2022-04-12,16,18.21,22.00,197,0.01,8.00,0.08
5,PC,Chrome,en,2022-04-12,2,7.52,28.00,109,0.01,4.00,0.02
6,PC,Firefox,de,2022-04-12,2,0.40,47.50,55,0.01,7.00,0.04
7,PC,Safari,en,2022-04-12,1,89.89,85.00,14,0.00,2.00,0.07
8,iOS,Safari,de,2022-04-12,10,0.83,15.00,260,0.01,11.00,0.04
9,iOS,Safari,en,2022-04-12,8,0.25,4.50,209,0.00,2.00,0.04


In [70]:
def compare_counts_and_cr_interactive(df):
    if df.empty or not all(col in df.columns for col in ['platform', 'browser', 'optimized_locale', 'count_ids_converted', 'count_ids_all_users', 'CR']):
        return

    df['group'] = df['platform'] + ' - ' + df['browser'] + ' - ' + df['optimized_locale']

    fig = go.Figure()

    fig.add_trace(go.Bar(x=df['group'], y=df['count_ids_converted'], name='Converted Users',
                         hovertemplate='Converted Users: %{y}<br>Group: %{x}<extra></extra>'))
    fig.add_trace(go.Bar(x=df['group'], y=df['count_ids_all_users'], name='All Users',
                         hovertemplate='All Users: %{y}<br>Group: %{x}<extra></extra>'))

    fig.add_trace(go.Scatter(x=df['group'], y=df['CR'], name='Conversion Rate', yaxis='y2',
                             mode='lines+markers', hovertemplate='Conversion Rate: %{y}<br>Group: %{x}<extra></extra>'))

    fig.update_layout(title='Comparison of User Counts and Conversion Rate Across Groups',
                      yaxis=dict(title='User Counts'),
                      yaxis2=dict(title='Conversion Rate', overlaying='y', side='right'),
                      xaxis_tickangle=-45)

    fig.show()


compare_counts_and_cr_interactive(merged_df)

- High Users, Low Conversions:

  -  Platforms like Android-Chrome-de and PC-Chrome-de have a large user base but a relatively low conversion rate.

  - This suggests UX, localization, or intent issues in these regions.

- High Conversion Rate, Low Users:

   - The iOS-Safari-pt group has a high conversion rate but very few total users.

   - This may indicate a niche, highly engaged audience.

- Android-Chrome-es performs better than most other Android groups in terms of conversion rate. This might suggest:

  - A more engaged Spanish-speaking audience.

  - Potential marketing success in that locale.

I face a very strange problem. Data for all sessions is present only for one day, so it is very hard to find trends in CR decline for latest dates in dataset. The only way left is to build a funnel for converted users and to check whether there was a decline in proportions in steps.

In [71]:
converted.head()


,final_id,context_locale,event_text,context_page_path,context_page_title,context_user_agent,timestamp,first_campaign,first_content,first_medium,first_source,first_time_order_completed,page_path_category,platform,browser,page_title_category,optimized_locale,campaign_category
0,202315551,de-DE,Filter,/subscribe,Fahrzeuge im Überblick | FINN,Mozilla/5.0 (iPhone; CPU iPhone OS 15_3_1 like...,2022-04-02 23:43:53.195000+00:00,S|DE|P|Subs|Intent|FINN,79597880,cpc,google,2022-03-24 19:31:18.353000+00:00,Subscribe - General,iOS,Safari,Vehicle Overview,de,Intent-Based Search
1,202315551,de-DE,Filter,/subscribe,Fahrzeuge im Überblick | FINN,Mozilla/5.0 (iPhone; CPU iPhone OS 15_3_1 like...,2022-04-02 23:43:53.195000+00:00,S|DE|P|Subs|Intent|FINN,79597880,cpc,google,2022-03-24 19:31:18.353000+00:00,Subscribe - General,iOS,Safari,Vehicle Overview,de,Intent-Based Search
2,202315551,de-DE,Product List Viewed,/subscribe,Fahrzeuge im Überblick | FINN,Mozilla/5.0 (iPhone; CPU iPhone OS 15_3_1 like...,2022-04-02 23:44:20.715000+00:00,S|DE|P|Subs|Intent|FINN,79597880,cpc,google,2022-03-24 19:31:18.353000+00:00,Subscribe - General,iOS,Safari,Vehicle Overview,de,Intent-Based Search
3,202315551,de-DE,Product List Viewed,/subscribe,Fahrzeuge im Überblick | FINN,Mozilla/5.0 (iPhone; CPU iPhone OS 15_3_1 like...,2022-04-02 23:44:20.715000+00:00,S|DE|P|Subs|Intent|FINN,79597880,cpc,google,2022-03-24 19:31:18.353000+00:00,Subscribe - General,iOS,Safari,Vehicle Overview,de,Intent-Based Search
4,209571451,ru,Checkout Step Viewed,/de-DE/checkout/thank_you/209571451/8552084246...,Bestellung | Vielen Dank für Ihre Bestellung,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,2022-04-07 23:44:13.235000+00:00,S|DE|P|Subs|DSA,Unknown Content,cpc,google,2022-04-07 23:28:18.708000+00:00,Checkout,PC,Safari,Order/Payment,ru,Other/Unclassified


In [87]:
def plot_funnels_over_time_rows_comparison(df, date_col='timestamp', event_col='event_text'):
    if df.empty or date_col not in df.columns or event_col not in df.columns:
        return

    df['date'] = pd.to_datetime(df[date_col]).dt.date
    df = df.dropna(subset=['date'])
    dates = sorted(df['date'].unique())

    if not dates:
        return

    num_dates = len(dates)

    def generate_funnel(filtered_df, title):
        events = filtered_df[event_col].value_counts().reset_index()
        events.columns = [event_col, 'count']

        # Filter events up to 'Order Completed' based on your image
        relevant_events = [
            'Product List Viewed', 'Filter', 'user-behaviour', 'Product Viewed',
            'UserAccount', 'Product Clicked', 'Checkout Step Viewed',
            'Checkout Step Completed', 'Product Added', 'additionalDrive',
            'Payment Info Entered', 'Checkout Started', 'Lead', 'Order Completed'
        ]
        events = events[events[event_col].isin(relevant_events)]

        fig = go.Figure(go.Funnel(y=events[event_col], x=events['count'], textinfo='value+percent initial'))
        fig.update_layout(title=title)
        fig.show()

    if num_dates == 1:
        generate_funnel(df[df['date'] == dates[0]], f"Funnel - {dates[0]}")
        return

    for date in dates:
        generate_funnel(df[df['date'] == date], f"Funnel - {date}")

    mid_index = num_dates // 2
    first_half_dates = dates[:mid_index]
    second_half_dates = dates[mid_index:]

    generate_funnel(df[df['date'].isin(first_half_dates)], "Funnel - First Half")
    generate_funnel(df[df['date'].isin(second_half_dates)], "Funnel - Second Half")



plot_funnels_over_time_rows_comparison(converted)

## Analysis Limitations:

Due to the limited dataset, particularly concerning the "all sessions" data, we face significant challenges in performing a comprehensive analysis. The primary issue is the **lack of temporal depth**; the "all sessions" table contains data for only a single day. This severely restricts our ability to:

* **Construct a meaningful Conversion Rate (CR):** A reliable CR requires tracking user behavior across a period of time. With only one day's worth of data, we cannot observe how CR fluctuates or identify any trends.
* **Detect anomalies:** Anomaly detection relies on comparing current data to historical patterns. Without a time series, we have no baseline for comparison.

Furthermore, analyzing trends solely within the "converted" user data presents its own set of limitations. While the generated funnel plots offer a glimpse into the converted user journey, they do not necessarily reflect the broader user behavior.

* **Limited representation:** Converted users represent a small subset of the total user base. Trends observed in this group may not be generalizable to all users.
* **Lack of context:** Without data from non-converted users, we lack the context needed to understand why some users convert and others do not. This makes it difficult to identify actionable insights for improving CR.

In essence, the absence of a time series in the "all sessions" data and the limited scope of the converted user data create a scenario where:

* We cannot reliably calculate and analyze CR.
* Anomaly detection is impractical.
* Trend analysis is confined to a potentially unrepresentative segment of users.

To overcome these limitations and conduct a more robust analysis, we require:

* **Time series data for all users:** Data spanning multiple days or weeks for both converted and non-converted users.
* **A broader perspective:** Data that encompasses the entire user journey, from initial interaction to conversion (or abandonment).

With a more comprehensive dataset, we could effectively calculate CR, identify anomalies, and uncover valuable insights that drive informed decisions.

# Task 2


In [73]:
task_2 = pd.read_csv('/content/drive/MyDrive/Finn/Dataset - Data Analyst Product - Task 2 - Experiment  - Aggregated Session Data.csv')

In [81]:
task_2.columns

Index(['customer_id', 'variation', 'country', 'user_type', 'platform',
       'device_type', 'fta_ua_type', 'had_plp', 'had_pdp', 'had_product_added',
       'had_lead_created', 'had_order_completed', 'cnt_sessions_plp',
       'cnt_sessions_pdp', 'cnt_sessions_product_added',
       'cnt_sessions_lead_created', 'cnt_sessions_order_completed', 'cnt_plp',
       'cnt_pdp', 'cnt_product_added', 'cnt_lead_created', 'sum_signed_value'],
      dtype='object')

In [74]:
task_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94398 entries, 0 to 94397
Data columns (total 22 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   customer_id                   94398 non-null  object
 1   variation                     94398 non-null  object
 2   country                       94398 non-null  object
 3   user_type                     94398 non-null  object
 4   platform                      94398 non-null  object
 5   device_type                   94398 non-null  object
 6   fta_ua_type                   94398 non-null  object
 7   had_plp                       94398 non-null  int64 
 8   had_pdp                       94398 non-null  int64 
 9   had_product_added             94398 non-null  int64 
 10  had_lead_created              94398 non-null  int64 
 11  had_order_completed           94398 non-null  int64 
 12  cnt_sessions_plp              94398 non-null  int64 
 13  cnt_sessions_pdp

No missing values, that is good, i assume `had_order_completed` signals about positive conversion

In [88]:
def perform_ab_test_and_group_analysis(df, variation_col, conversion_col, group_cols, alpha=0.05):
    """Performs A/B test and analyzes conversion by groups, including plots."""

    if df.empty or variation_col not in df.columns or conversion_col not in df.columns:
        return {"error": "DataFrame is empty or missing required columns."}

    variations = df[variation_col].unique()

    if len(variations) != 2:
        return {"error": "Exactly two variations are required for an A/B test."}

    # Overall A/B test
    print("Overall A/B Test Results:")
    overall_results = perform_ab_test_internal(df, variation_col, conversion_col, alpha)

    # Plot overall conversion rates
    plot_conversion_rates(df, variation_col, conversion_col, "Overall Conversion Rates")

    # Analyze by groups and plot
    for group_col in group_cols:
        print(f"\nAnalysis by {group_col}:")
        for group in df[group_col].unique():
            group_df = df[df[group_col] == group]
            print(f"  Group: {group}")
            if len(group_df[variation_col].unique()) == 2:
                perform_ab_test_internal(group_df, variation_col, conversion_col, alpha)
                plot_conversion_rates(group_df, variation_col, conversion_col, f"Conversion Rates - {group_col}: {group}")
            else:
                print("    Skipping: Only one variation present in this group.")

def perform_ab_test_internal(df, variation_col, conversion_col, alpha):
    """Internal function to perform the A/B test."""
    group_a = df[df[variation_col] == df[variation_col].unique()[0]][conversion_col]
    group_b = df[df[variation_col] == df[variation_col].unique()[1]][conversion_col]

    if all(isinstance(x, (int, float)) for x in df[conversion_col]):
        t_stat, p_value = stats.ttest_ind(group_a, group_b)
        print(f"    T-statistic: {t_stat:.4f}, P-value: {p_value:.4f}")
        if p_value < alpha:
            print(f"    Statistically significant (p < {alpha}).")
        else:
            print(f"    Not statistically significant (p >= {alpha}).")
        return {"t_stat": t_stat, "p_value": p_value}
    else:
        contingency_table = pd.crosstab(df[variation_col], df[conversion_col])
        chi2, p_value, _, _ = stats.chi2_contingency(contingency_table)
        print(f"    Chi-squared: {chi2:.4f}, P-value: {p_value:.4f}")
        if p_value < alpha:
            print(f"    Statistically significant (p < {alpha}).")
        else:
            print(f"    Not statistically significant (p >= {alpha}).")
        return {"chi2": chi2, "p_value": p_value}

def calculate_conversion_rate(df, variation_col, conversion_col):
    """Calculates conversion rates for each variation."""
    if df.empty or variation_col not in df.columns or conversion_col not in df.columns:
        return {"error": "DataFrame is empty or missing required columns."}

    conversion_rates = {}
    for variation in df[variation_col].unique():
        variation_df = df[df[variation_col] == variation]
        conversion_rates[variation] = variation_df[conversion_col].mean()

    print("\nConversion Rates:")
    for variation, rate in conversion_rates.items():
        print(f"{variation}: {rate:.4f}")

    return conversion_rates

def plot_conversion_rates(df, variation_col, conversion_col, title):
    """Plots conversion rates for each variation."""
    rates = calculate_conversion_rate(df, variation_col, conversion_col)
    if "error" in rates:
        return

    variations = list(rates.keys())
    values = list(rates.values())

    fig = go.Figure(data=[go.Bar(x=variations, y=values)])
    fig.update_layout(title=title)
    fig.show()

group_cols = ['country', 'user_type', 'platform', 'device_type', 'fta_ua_type']
perform_ab_test_and_group_analysis(task_2, 'variation', 'had_order_completed', group_cols)
calculate_conversion_rate(task_2, 'variation', 'had_order_completed')


Overall A/B Test Results:
    T-statistic: -1.1795, P-value: 0.2382
    Not statistically significant (p >= 0.05).

Conversion Rates:
a: 0.0061
b: 0.0067



Analysis by country:
  Group: DE
    T-statistic: -1.1795, P-value: 0.2382
    Not statistically significant (p >= 0.05).

Conversion Rates:
a: 0.0061
b: 0.0067



Analysis by user_type:
  Group: recurring
    T-statistic: -1.4190, P-value: 0.1559
    Not statistically significant (p >= 0.05).

Conversion Rates:
a: 0.0153
b: 0.0177


  Group: new
    T-statistic: 0.2537, P-value: 0.7997
    Not statistically significant (p >= 0.05).

Conversion Rates:
b: 0.0031
a: 0.0030



Analysis by platform:
  Group: mobile web
    T-statistic: -0.8391, P-value: 0.4014
    Not statistically significant (p >= 0.05).

Conversion Rates:
a: 0.0040
b: 0.0044


...

  Group: Brand
    T-statistic: 1.3966, P-value: 0.1653
    Not statistically significant (p >= 0.05).

Conversion Rates:
a: 0.0638
b: 0.0149


  Group: Ops Customer Product
    T-statistic: nan, P-value: nan
    Not statistically significant (p >= 0.05).

Conversion Rates:
a: 0.0000
b: 0.0000


  Group: Ops Customer Success
    Skipping: Only one variation present in this group.
  Group: B2B Brand
    T-statistic: nan, P-value: nan
    Not statistically significant (p >= 0.05).

Conversion Rates:
b: 0.0000
a: 0.0000



Conversion Rates:
a: 0.0061
b: 0.0067


{'a': np.float64(0.006051373196225297), 'b': np.float64(0.006661575016972165)}

## A/B Test Results Analysis

**Overall A/B Test Results:**

* **T-statistic:** -1.1795
* **P-value:** 0.2382
* **Conclusion:** Not statistically significant (p >= 0.05).

**Analysis by Country:**

* **Group: DE**
    * **T-statistic:** -1.1795
    * **P-value:** 0.2382
    * **Conclusion:** Not statistically significant (p >= 0.05).

**Analysis by User Type:**

* **Group: recurring**
    * **T-statistic:** -1.4190
    * **P-value:** 0.1559
    * **Conclusion:** Not statistically significant (p >= 0.05).
* **Group: new**
    * **T-statistic:** 0.2537
    * **P-value:** 0.7997
    * **Conclusion:** Not statistically significant (p >= 0.05).

**Analysis by Platform:**

* **Group: mobile web**
    * **T-statistic:** -0.8391
    * **P-value:** 0.4014
    * **Conclusion:** Not statistically significant (p >= 0.05).
* **Group: web**
    * **T-statistic:** 1.3579
    * **P-value:** 0.1745
    * **Conclusion:** Not statistically significant (p >= 0.05).
* **Group: app**
    * **T-statistic:** -0.7631
    * **P-value:** 0.4455
    * **Conclusion:** Not statistically significant (p >= 0.05).

**Analysis by Device Type:**

* **Group: Mobile**
    * **T-statistic:** -0.4581
    * **P-value:** 0.6469
    * **Conclusion:** Not statistically significant (p >= 0.05).
* **Group: Computer**
    * **T-statistic:** 1.2977
    * **P-value:** 0.1944
    * **Conclusion:** Not statistically significant (p >= 0.05).
* **Group: Other**
    * **T-statistic:** 0.7015
    * **P-value:** 0.4831
    * **Conclusion:** Not statistically significant (p >= 0.05).
* **Group: Tablet**
    * **T-statistic:** -1.3159
    * **P-value:** 0.1909
    * **Conclusion:** Not statistically significant (p >= 0.05).

**Analysis by fta_ua_type:**

* **Group: Performance**
    * **T-statistic:** -0.9633
    * **P-value:** 0.3354
    * **Conclusion:** Not statistically significant (p >= 0.05).
* **Group: Direct**
    * **T-statistic:** 0.5344
    * **P-value:** 0.5931
    * **Conclusion:** Not statistically significant (p >= 0.05).
* **Group: Partnerships**
    * **T-statistic:** 0.6084
    * **P-value:** 0.5429
    * **Conclusion:** Not statistically significant (p >= 0.05).
* **Group: no_data**
    * **T-statistic:** 1.2822
    * **P-value:** 0.2000
    * **Conclusion:** Not statistically significant (p >= 0.05).
* **Group: Organic**
    * **T-statistic:** -1.6125
    * **P-value:** 0.1069
    * **Conclusion:** Not statistically significant (p >= 0.05).
* **Group: Customer Growth**
    * **T-statistic:** -1.6904
    * **P-value:** 0.0912
    * **Conclusion:** Not statistically significant (p >= 0.05).
* **Group: Internal**
    * **T-statistic:** 1.0727
    * **P-value:** 0.2841
    * **Conclusion:** Not statistically significant (p >= 0.05).
* **Group: B2B Performance**
    * **T-statistic:** 0.5154
    * **P-value:** 0.6068
    * **Conclusion:** Not statistically significant (p >= 0.05).
* **Group: Brand**
    * **T-statistic:** 1.3966
    * **P-value:** 0.1653
    * **Conclusion:** Not statistically significant (p >= 0.05).
* **Group: Ops Customer Product**
    * **T-statistic:** NaN
    * **P-value:** NaN
    * **Conclusion:** Not statistically significant (p >= 0.05).
* **Group: Ops Customer Success**
    * **Conclusion:** Skipped: Only one variation present in this group.
* **Group: B2B Brand**
    * **T-statistic:** NaN
    * **P-value:** NaN
    * **Conclusion:** Not statistically significant (p >= 0.05).

**Conversion Rates:**

* **a:** 0.0061
* **b:** 0.0067

**Key Findings:**

* No statistically significant difference in conversion rates between variations 'a' and 'b' was observed, either overall or within any subgroups.
* Conversion rates are generally low (around 0.6%).
* Some subgroups showed fluctuations, that while not statistically significant, may warrant investigation.
* Groups with `NaN` values, should be investigated.

**Recommendations:**

* Review experiment setup for errors.
* Consider larger sample sizes for increased statistical power.
* Investigate user experience to identify areas for improvement.
* Explore other factors influencing conversion.
* Investigate subgroups with `NaN` values.